# wrap_tool_call钩子
## 1.基于装饰器实现

In [1]:
from langchain.messages import HumanMessage,ToolMessage
from langchain.agents.middleware import AgentMiddleware,wrap_tool_call,ToolCallRequest
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from typing import Callable
from langchain.agents import create_agent
from langgraph.types import Command
from rich import print as rprint
from typing import Any

@tool
def get_weather(city: str, is_forcast: bool) -> str:
    """
    获取当日特定城市的天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明天的天气预报
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天天气也很好"
    return res


@wrap_tool_call
def wrap_tool_call_middleware(request:ToolCallRequest,handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],) -> ToolMessage | Command[Any]:
    result = handler(request)
    print(f"原始参数：{request.tool_call['args']}")
    print(f"原始参数调用结果：{result}")

    # 修改工具参数的值
    request.tool_call['args']['is_forcast'] = True
    result = handler(request)
    print(f"修改后参数：{request.tool_call['args']}")
    print(f"修改后调用结果：{result}")

    return result

# 1.初始化模型
model = ChatOpenAI(
    # model="qwen3-vl:latest",
    model="483025889/qwen3.5:9b", # ok
    # model="gemma4:12b", # 这里不好用
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

# 2.创建agent
agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
     wrap_tool_call_middleware,
    ]
)

resp = agent.invoke({"messages":[HumanMessage("帮我查询北京今天的天气如何？")]})
for msg in resp['messages']:
    msg.pretty_print()

原始参数：{'city': '北京', 'is_forcast': False}
原始参数调用结果：content='北京今天天气不错' name='get_weather' tool_call_id='call_eib6ha6t'
修改后参数：{'city': '北京', 'is_forcast': True}
修改后调用结果：content='北京今天天气不错\n明天天气也很好' name='get_weather' tool_call_id='call_eib6ha6t'
================================ Human Message =================================

帮我查询北京今天的天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_eib6ha6t)
 Call ID: call_eib6ha6t
  Args:
    city: 北京
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

北京今天天气不错
明天天气也很好
================================== Ai Message ==================================

北京今天的天气很不错哦，另外明天的天气预报显示天气也会很好～


## 基于类实现

In [2]:
from langchain.messages import HumanMessage,ToolMessage
from langchain.agents.middleware import AgentMiddleware,wrap_tool_call,ToolCallRequest
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from typing import Callable
from langchain.agents import create_agent
from langgraph.types import Command
from rich import print as rprint
from typing import Any

@tool
def get_weather(city: str, is_forcast: bool) -> str:
    """
    获取当日特定城市的天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明天的天气预报
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天天气也很好"
    return res


class WrapToolCallMiddleware(AgentMiddleware):
    def wrap_tool_call(self,request:ToolCallRequest,handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],) -> ToolMessage | Command[Any]:
        result = handler(request)
        print(f"原始参数：{request.tool_call['args']}")
        print(f"原始参数调用结果：{result}")

        # 修改工具参数的值
        request.tool_call['args']['is_forcast'] = True
        result = handler(request)
        print(f"修改后参数：{request.tool_call['args']}")
        print(f"修改后调用结果：{result}")

        return result

# 1.初始化模型
model = ChatOpenAI(
    # model="qwen3-vl:latest",
    model="483025889/qwen3.5:9b", # ok
    # model="gemma4:12b", # 这里不好用
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

# 2.创建agent
agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
     WrapToolCallMiddleware(),
    ]
)

resp = agent.invoke({"messages":[HumanMessage("帮我查询北京今天的天气如何？")]})
for msg in resp['messages']:
    msg.pretty_print()

原始参数：{'city': '北京', 'is_forcast': False}
原始参数调用结果：content='北京今天天气不错' name='get_weather' tool_call_id='call_8xx2cua2'
修改后参数：{'city': '北京', 'is_forcast': True}
修改后调用结果：content='北京今天天气不错\n明天天气也很好' name='get_weather' tool_call_id='call_8xx2cua2'
================================ Human Message =================================

帮我查询北京今天的天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_8xx2cua2)
 Call ID: call_8xx2cua2
  Args:
    city: 北京
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

北京今天天气不错
明天天气也很好
================================== Ai Message ==================================

北京今天天气不错。
